In [1]:
# ============================================================
# Cell 1: Import Required Libraries
# ============================================================

import time
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from torchvision.models import (
    efficientnet_b2,
    EfficientNet_B2_Weights
)

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score

print("Cell 1 : Libraries Imported Successfully")

Cell 1 : Libraries Imported Successfully


In [2]:
# ============================================================
# Cell 2: Configuration
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_SIZE = 260
BATCH_SIZE = 16
ABLATION_EPOCHS = 10       # quick run, not full training
LEARNING_RATE = 1e-4
NUM_CLASSES = 3
NUM_WORKERS = 0

CLASS_NAMES = ["BACTERIA", "NORMAL", "VIRUS"]

PROJECT_ROOT = Path(
    "/mnt/g/Research paper/Research paper/Pneumonia-MultiModel-XAI"
)
DATASET_DIR = PROJECT_ROOT / "dataset" / "processed_dataset"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ABLATION_RESULTS_PATH = RESULTS_DIR / "ablation_study_results.csv"

print(f"Device : {DEVICE}")
print(f"Ablation Epochs per variant : {ABLATION_EPOCHS}")

Device : cuda
Ablation Epochs per variant : 10


In [3]:
# ============================================================
# Cell 3: Data Preparation
# ============================================================

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.90, 1.10)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.10)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

valid_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(DATASET_DIR / "train", transform=train_transform)
valid_dataset = datasets.ImageFolder(DATASET_DIR / "validation", transform=valid_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_dataset.targets),
    y=train_dataset.targets
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

print(f"Train : {len(train_dataset)}  |  Valid : {len(valid_dataset)}")

Train : 4099  |  Valid : 878
